In [2]:

# # 03 - QSVM Experiments
#
# This notebook implements the Quantum Support Vector Machine (QSVM) using Qiskit
# on the preprocessed breast cancer dataset. We will define the quantum feature map,
# create the quantum kernel, train the QSVC model, and evaluate its performance.

# %%
# === Step 1: Import Necessary Libraries ===

import numpy as np
import matplotlib.pyplot as plt
import time # To measure training time
# Scikit-learn metrics
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_curve, auc

# Qiskit libraries
# Aer provider import differs after Qiskit 1.0+
# Assuming current Qiskit version (1.0+):
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from qiskit.primitives import Sampler
from qiskit_aer import AerSimulator


# Set seed for reproducibility
seed = 128
np.random.seed(seed)

In [ ]:

# ## Step 2: Load Processed Data
#
# We load the training, validation, and test sets that were saved in the
# `01_data_exploration_and_preprocessing.ipynb` notebook. We assume these files
# are located in the `data/processed` folder.
# %%
# Load data from saved NumPy files

try:
    X_train = np.load('data/processed/X_train.npy')
    y_train = np.load('data/processed/y_train.npy')
    X_val = np.load('data/processed/X_val.npy') # Validation set (optional for QSVM tuning)
    y_val = np.load('data/processed/y_val.npy')
    X_test = np.load('data/processed/X_test.npy')
    y_test = np.load('data/processed/y_test.npy')

    # Check shapes
    print("X_train shape:", X_train.shape)
    print("X_test shape:", X_test.shape)
    print("y_train shape:", y_train.shape)

    # Get the number of features (qubits) from X_train
    n_features = X_train.shape[1]
    print(f"Data reduced to {n_features} features.")

except FileNotFoundError:
    print("ERROR: Processed data files (data/processed/*.npy) not found.")
    print("Please run the '01_data_exploration_and_preprocessing.ipynb' notebook first.")
    # Stop execution if data is missing
    # raise SystemExit("Stopping execution due to missing data.")

In [ ]:

# ## Step 3: Define Quantum Feature Map and Kernel
#
# We define the feature map that encodes classical data into quantum states,
# and the quantum kernel that measures the similarity between these states.

# %%
# === Quantum Parameters ===
feature_dimension = n_features # Number of features after PCA
reps = 2 # Number of times to repeat the feature map circuit (increases complexity)
entanglement = 'linear' # Entanglement structure ('linear', 'full', 'circular'...)

# === Feature Map ===
# Let's use ZZFeatureMap
# Feature map oluşturma
feature_map = ZZFeatureMap(feature_dimension, reps)

print("Feature Map Used:")
# We can draw the circuit (readable for small number of qubits)
if n_features <= 5:
     try:
         # display is specific to Jupyter environments
         from IPython.display import display
         display(feature_map.decompose().draw('mpl', style='iqx')) # IQX style
     except Exception as e:
         print(f"Could not draw circuit: {e}")
         print(feature_map) # Print as text
else:
     print(f"ZZFeatureMap for {n_features} qubits (reps={reps}, entanglement='{entanglement}')")

# === Quantum Kernel ===
# Qiskit 1.0+ recommends using Sampler primitive

# Sampler ve Fidelity quantum kernel oluşturma
sampler = Sampler(AerSimulator())
fidelity_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=sampler)


print("\nQuantum Kernel created successfully.")

# %% [markdown]
# ## Step 4: Create and Train the QSVM Model
#
# We instantiate the QSVC model using our defined quantum kernel and train it
# with the training data.

# %%
# === Create QSVC Model ===
# QSVC modelini oluşturma ve eğitme
qsvc = QSVC(quantum_kernel=fidelity_kernel)

# === Train the Model ===
print("Starting QSVM Training...")
start_time = time.time() # Start time to measure training duration

qsvc.fit(X_train, y_train)

end_time = time.time() # End time
training_time = end_time - start_time
print(f"Training Complete. Duration: {training_time:.2f} seconds")

In [ ]:
# ## Step 5: Evaluate the Model
#
# We measure the performance of our trained QSVM model on the unseen test set.

# %%
# === Make Predictions on Test Set ===
print("\nPredicting on the test set...")
y_pred = qsvc.predict(X_test)

# === Calculate Performance Metrics ===
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"\nQSVM Test Accuracy: {accuracy:.4f}")

print("\nConfusion Matrix:")
print(conf_matrix)

print("\nClassification Report:")
print(class_report)

# === Visualizations ===

# Plot Confusion Matrix
try:
    disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("QSVM Confusion Matrix")
    plt.show()
except NameError: # Handle case where ConfusionMatrixDisplay might not be imported if sklearn version is old
     print("Could not plot confusion matrix. Ensure scikit-learn version is up-to-date.")


# Plot ROC Curve
# ROC typically requires probability scores. QSVC predict gives direct classes.
# Decision function can be used for a score, but it's not a probability in SVMs.
# Let's focus on accuracy and classification report for now.
# Advanced: Check if QSVC has probability=True similar to SVC, or use kernel matrix
# with SVC(kernel='precomputed', probability=True).

print("\nEvaluation complete.")